# Week 1 Day 2 - The cheapest agent that gets 15 / 25 right

The challenge: using the agent from the lab, find the cheapest model that gets at least 15 of the 25 claims right, consistently.

My approach:

1. Pick every model on OpenRouter that supports tool calling and lists cheaper than Luna ($0.10 in / $0.50 out per million tokens).
2. Run each one through the lab's agent once, on the whole queue. This is the screen.
3. Rerun the models that pass the screen three times, to check they pass every time.
4. Rank by the cost OpenRouter actually charged, not by the list price.

The agent, prompt, tools and scoring are the same as the lab's.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

DATA = Path("../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("evals/decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

## The agent

The same instructions and tools as the lab. One change: the lab keeps decisions in one global `recorded` dict, but I run models side by side, so each agent gets its own `recorded` dict and its own tools.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""


def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def make_agent(model, effort, recorded):
    """An agent on this model that records its decisions in `recorded`."""

    @tool
    def get_policy(policy_id: str) -> str:
        """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
        policy = policies.get(policy_id)
        return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"

    @tool
    def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
        """Record the decision on a claim in the claims system.

        payout is the amount to pay in US dollars, or 0 for a decline or a referral.
        rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
        """
        recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
        return f"Recorded {decision} on {claim_id}"

    # Many cheap models don't take reasoning_effort, so it's only sent when set
    extra = {"reasoning_effort": effort} if effort else {}
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=3, **extra)
    return create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

## Running the queue

`run_queue` runs all 25 claims, a few at a time, and adds up OpenRouter's cost for every model call the agent made. A claim with no recorded decision counts as wrong, as in the lab.

Two things I hit along the way:

- New OpenRouter accounts have a requests-per-minute limit. Luna's first runs lost claims to `429` errors, which made it look worse than it is. A claim that's rate-limited now waits and tries again.
- Some models loop on tool calls. `recursion_limit` stops a claim after 12 steps, and it counts as wrong.

In [ ]:
def run_queue(model, effort=None, workers=3):
    """Run the agent on every claim and return the score, the cost and the time."""
    recorded = {}
    agent = make_agent(model, effort, recorded)

    def decide(claim):
        cost = 0.0
        prompt = f"# Claim\n\n{json.dumps(claim, indent=2)}\n"
        for _ in range(6):
            try:
                result = agent.invoke({"messages": [{"role": "user", "content": prompt}]}, {"recursion_limit": 12})
                cost += sum((message.response_metadata.get("token_usage") or {}).get("cost") or 0 for message in result["messages"] if message.type == "ai")
                break
            except Exception as error:
                if "RateLimit" not in type(error).__name__:
                    break
                time.sleep(30)
        answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0})
        return is_right(answer, labels[claim["claim_id"]]), cost

    start = time.time()
    with ThreadPoolExecutor(workers) as pool:
        results = list(pool.map(decide, claims))
    row = {
        "model": model,
        "effort": effort or "default",
        "right": sum(right for right, _ in results),
        "cost": sum(cost for _, cost in results),
        "minutes": round((time.time() - start) / 60, 1),
    }
    print(row)
    return row

## Screen: one run each

These are the tool-calling models on OpenRouter that list cheaper than Luna, plus Luna itself. Running them all takes a while and costs about 30 cents.

In [ ]:
candidates = [
    ("openai/gpt-6-luna", "low"),
    ("openai/gpt-oss-20b", "low"),
    ("inclusionai/ling-3.0-flash", None),
    ("mistralai/mistral-nemo", None),
    ("qwen/qwen3.7-flash", None),
    ("amazon/nova-micro-v1", None),
    ("inception/mercury-2.5", None),
    ("meta-llama/llama-3.1-8b-instruct", None),
    ("openai/gpt-5-nano", "low"),
    ("deepseek/deepseek-v4.1-flash", None),
    ("nvidia/nemotron-3.5-lightning", None),
    ("openai/gpt-oss-120b", "low"),
    ("mistralai/mistral-small-3.2-24b-instruct", None),
    ("google/gemini-2.5-flash-lite", None),
]

screen = [run_queue(model, effort) for model, effort in candidates]
pd.DataFrame(screen).sort_values("cost")

## Consistency: three more runs of the finalists

Luna runs one claim at a time here to stay under the new-account rate limit.

In [ ]:
finalists = [("openai/gpt-6-luna", "low", 1), ("inclusionai/ling-3.0-flash", None, 3)]

repeats = [run_queue(model, effort, workers) for model, effort, workers in finalists for _ in range(3)]
pd.DataFrame(repeats).groupby(["model", "effort"])[["right", "cost", "minutes"]].agg(["min", "mean", "max"])

## Two last ideas

Can Luna get cheaper with no reasoning? Can the cheapest model, gpt-oss-20b, pass if it reasons more?

In [ ]:
extras = [run_queue("openai/gpt-6-luna", "none", workers=1), run_queue("openai/gpt-oss-20b", "medium")]

## My results

The cost of one run of the 25-claim queue, as charged by OpenRouter.

| Model | Effort | Right | Cost a run | Time a run |
|---|---|---|---|---|
| **openai/gpt-6-luna** | **low** | **17, 16, 18** | **$0.0065** | **4 min** |
| inclusionai/ling-3.0-flash | default | 16, 15, 15, 16 | $0.0068 | 18 min |
| openai/gpt-6-luna | none | 14 | $0.0077 | 4 min |
| openai/gpt-oss-20b | low | 11 | $0.0041 | 2 min |
| openai/gpt-oss-20b | medium | 12 | $0.0069 | 9 min |
| deepseek/deepseek-v4.1-flash | default | 18 | $0.0585 | 1 min |
| qwen/qwen3.7-flash | default | 15 | $0.0182 | 6 min |
| nvidia/nemotron-3.5-lightning | default | 15 | $0.0439 | 3 min |
| inception/mercury-2.5 | default | 14 | $0.0149 | 1 min |
| openai/gpt-oss-120b | low | 10 | $0.0052 | 1 min |
| openai/gpt-5-nano | low | 8 | $0.0179 | 1 min |
| mistralai/mistral-nemo | default | 6 | $0.0024 | 1 min |
| amazon/nova-micro-v1 | default | 5 | $0.0119 | under 1 min |
| google/gemini-2.5-flash-lite | default | 5 | $0.0175 | under 1 min |
| mistralai/mistral-small-3.2-24b-instruct | default | 5 | $0.0171 | 1 min |
| meta-llama/llama-3.1-8b-instruct | default | 1 | $0.0109 | under 1 min |

z-ai/glm-5.3-flash and qwen/qwen3.5-9b hadn't finished a run after an hour, so I left them out.

## What I learned

- **The answer is Luna at `low`.** It passed on all three runs, 16 to 18 right, for about $0.0065 a run. Nothing that costs less got to 15.
- **The list price is not the cost.** Twelve of these models list cheaper than Luna, and only one of them, Ling, passed. Ling still cost a little more than Luna in practice and took four times as long. DeepSeek lists at about half Luna's price but cost nine times as much, because it used far more tokens on each claim.
- **Less reasoning didn't save money.** Luna at `none` got fewer right and cost more, presumably because it took more steps to reach a decision.
- **Small models struggle with the tools.** Most of the tiny models often never called `record_decision`, so those claims scored as wrong.
- **Check the rate limits before trusting a score.** Luna's first run scored 7 because a new account's rate limit refused the calls, not because the model was wrong.